# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Metadata exploration
md = dataset.metadata
print(f"Dataset Title: {md.name}\nDescription: {md.description}\n")
print(f"Version: {md.version}\nPublished: {md.datePublished}\nLicense: {md.license}\n")
print(f"Keywords: {md.keywords}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references will use their `@id` field.

In [ ]:
# Show all available record sets with `@id` and name
print("Record sets in the dataset:")
for rs in md.record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For this dataset, let's choose the main record set by its @id (commonly 'cr:main' or similar, here we need to inspect the available record sets)
record_sets = [rs['@id'] for rs in md.record_sets]
print("\nAll record set @ids:", record_sets)

# For each record set, show its fields and their @ids
for rs in md.record_sets:
    print(f"\nRecord set '@id': {rs['@id']}")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
    else:
        print("  No fields defined.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references use their `@id` as given in the previous section.

> **Note:** You can reference any record set and field by their `@id`. For this notebook, we focus on the main analysis table. If the dataset contains one primary record set (e.g. the table of protein abundances), use its `@id` below. For illustration, we'll assume a common main data `@id` (replace as appropriate after inspecting previous output).

In [ ]:
# If you inspected above, set the 'main_record_set_id' accordingly. For this FAIR2 dataset, we will typically find one main table (otherwise, adjust as needed).
# For example, suppose the main data record set is '@id': 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json#ms_abundances' (dummy example for illustration).
# Use the printed @id from the previous step!

# Set your chosen record set(s) explicitly:
main_record_set_id = None
for rs in md.record_sets:
    if ('abundance' in rs.get('name', '').lower() or 'protein' in rs.get('name', '').lower()):
        main_record_set_id = rs['@id']
        break
if not main_record_set_id:
    main_record_set_id = record_sets[0] # fallback to the first
print(f"Using main record set: {main_record_set_id}")

# Extract data
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Record set {record_set_id}: {df.shape[0]} rows, {df.shape[1]} cols")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# Display columns from the main record set
df = dataframes[main_record_set_id]
print("Available columns (field @ids):")
print(df.columns.tolist())

df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations here reference fields by their `@id` (i.e., as column names in the DataFrame loaded earlier).

In [ ]:
# Choose a numeric field for analysis (using its @id as in df.columns)
# For demonstration, let's pick a common field name such as 'coverage' or 'abundance'.

# Display all columns for manual inspection
print("Main DataFrame columns:", df.columns.tolist())

# Choose a numeric field based on column names (update as needed):
candidate_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['abundance', 'coverage', 'count', 'mw', 'intensity'])]
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.select_dtypes(include=[float, int]).columns[0]
print(f"Using numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
print(f"Using threshold: {threshold}")

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize this numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a field (e.g., by protein description if available)
candidate_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['description', 'accession', 'category', 'sample'])]
group_field = candidate_group_fields[0] if candidate_group_fields else None

if group_field is not None and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped data is available, show top groups
if 'grouped_df' in locals():
    grouped_df.sort_values(numeric_field, ascending=False).head(10).plot.bar(
        figsize=(10, 4), legend=False, title=f"Top 10 groups by mean {numeric_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we:
- Explored the metadata and structure of the FAIR² dataset using the `mlcroissant` library.
- Inspected record set and field `@id`s and loaded records programmatically.
- Performed basic exploratory data analysis including normalization and group aggregation based on field `@id`s (column names).
- Visualized the distribution of measured values.

You can adapt the analysis by modifying the chosen record set and field `@id`s based on your research question or data needs.